# MIRAGE Quantum Machine Learning for Traffic Classification
di Mario Gabriele Carofano

### Panoramica

### Dataset

### Output


---

In [ ]:
#	LIBRARIES
#   ####################################################################    #

# Importing constant values
import constants

# Data loading and saving
import pickle
import os

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Data preprocessing and evaluation
from preprocessing_functions import *
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Machine Learning and Quantum ML
from training_functions import *
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import pennylane as qml

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Other utilities
import copy
import datetime
import random
import time

In [ ]:
#	MACROS
#   ####################################################################    #

import importlib
importlib.reload(constants)
from constants import RANDOM_SEED

#   ####################################################################    #

# 1. Configurazione del seed per la riproducibilità.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED) # Utile per hash di dizionari/set

# 2. Configurazione di PyTorch (CPU e GPU)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

# 3. Configurazione di PyTorch per la riproducibilità su GPU (CUDNN)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 4. Configurazione del dispositivo (CPU o GPU)
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
# DEVICE = 'cpu'
print(f"Using device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# CLASSES
#   ####################################################################    #

class MirageDataset(Dataset):
	""" Custom PyTorch Dataset per il caricamento e
	la preparazione dei dati di traffico di rete.
	
	Trasforma i dati da formato (N, Packets, Features)
	a formato (N, Features, Packets) per compatibilità con
	reti neurali convoluzionali 1D (CNN1D).
	"""
		
	def __init__(self, X, y):
		"""Inizializza il dataset Mirage.

		Args:
			X (torch.Tensor): Dati di input, shape (N, Packets, Features).
			y (torch.Tensor): Etichette corrispondenti, shape (N,).
		"""

		if DEVICE == 'mps':
			self.X = torch.FloatTensor(X).permute(0, 2, 1).float().to(DEVICE)
		else:
			self.X = torch.FloatTensor(X).permute(0, 2, 1).double().to(DEVICE)

		self.y = torch.LongTensor(y).to(DEVICE)

		# end

	def __len__(self):
		"""Ritorna la lunghezza del dataset.

		Returns:
			int: Lunghezza del dataset.
		"""

		return len(self.y)
	
		# end

	def __getitem__(self, idx):
		"""Ritorna un elemento del dataset.

		Args:
			idx (int): Indice dell'elemento da recuperare.

		Returns:
			tuple: Coppia (X, y) dell'elemento.
		"""
		
		return self.X[idx], self.y[idx]
	
		# end
	
	# end class

---

## Caricamento dei dati

Caricamento del file pickle preprocessato contenente biflussi e etichette.

In [ ]:
import importlib
importlib.reload(constants)
from constants import DATA_PATH

#   ####################################################################    #

print("Loading dataset...")

# Il file pickle contiene due oggetti salvati in sequenza:
# 1. X_raw : i dati numerici dei flussi di traffico, in formato numpy array.
# 2. y_raw : le etichette corrispondenti ai flussi.
# Ogni chiamata a pickle.load() legge il successivo oggetto nel file.
with open(DATA_PATH, "rb") as f:
    X_raw = np.array(pickle.load(f), dtype=np.float32)
    y_raw = np.array(pickle.load(f))

print(f"Data shape: {X_raw.shape}")
print(f"Labels shape: {len(y_raw)}")

# Si definisce un campione d'esempio per tutto il notebook.
example_sample = 7000
print("\nExample Sample (First 5 packets):\n", X_raw[example_sample][:5])
print("Example Label:", y_raw[example_sample])

## Label Encoding

Trasformazione delle etichette categoriche in valori numerici utilizzando `LabelEncoder`.

Ogni classe di traffico viene associata a un identificativo numerico, facilitando l'utilizzo nei modelli ML. Le classi originali vengono memorizzate in `DATASET_CLASSES` per riferimenti futuri, mentre il numero totale di classi viene salvato in `NUM_CLASSES`.

In [ ]:
le = LabelEncoder()

y_encoded = le.fit_transform(y_raw)

DATASET_CLASSES = le.classes_
""" List of unique classes in the dataset, determined by the unique labels in y_raw after encoding. """

NUM_CLASSES = len(le.classes_)
""" Number of unique classes in the dataset. """

print(
    f"Number of classes: {NUM_CLASSES} \n\n" +
	f"Classes: {DATASET_CLASSES}"
)

## Train-Validation-Test split

Il dataset viene suddiviso in tre insiemi distinti mediante doppio split stratificato, le cui dimensioni sono specificate dalle costanti `TRAIN_SIZE`, `VAL_SIZE` e `TEST_SIZE`, preservando la distribuzione originale delle classi.

Se il training set supera `NEW_TRAIN_SIZE` campioni, viene ridotto tramite campionamento casuale per contenere i tempi di addestramento.

In [ ]:
import importlib
importlib.reload(constants)
from constants import (
	TRAIN_SIZE, VAL_SIZE, TEST_SIZE, NEW_TRAIN_SIZE,
    RANDOM_SEED
)

#   ####################################################################    #

# Verifica che le proporzioni di Train, Val e Test sommino a "1".
assert TRAIN_SIZE + VAL_SIZE + TEST_SIZE == 1.0, "Train, Val, Test sizes must sum to 1."

# Il primo split divide il dataset in due parti: Train+Val e Test.
X_temp, X_test, y_temp, y_test = train_test_split(
    X_raw, y_encoded,
    test_size=TEST_SIZE,
    stratify=y_encoded,
    random_state=RANDOM_SEED
)

# Il fitting dello scaler viene fatto solo sul Train+Val, per evitare data leakage.
# ...

# Il secondo split divide il Train+Val rimanente in due insiemi: Train e Val.
tmp_size = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=tmp_size,
    stratify=y_temp,
    random_state=RANDOM_SEED
)

print(
	f"Train shape: {X_train.shape} ({len(X_train)/len(X_raw):.1%}) \n" +
	f"Val shape:   {X_val.shape}   ({len(X_val)/len(X_raw):.1%}) \n" +
	f"Test shape:  {X_test.shape}  ({len(X_test)/len(X_raw):.1%}) \n"
)

# Riduzione della dimensione del training set, se necessario.
if len(X_train) > NEW_TRAIN_SIZE:
    
	# Calcola la frazione del training set da mantenere per ridurlo a NEW_TRAIN_SIZE
	percentage = NEW_TRAIN_SIZE / len(X_train)
    
	X_train, _, y_train, _ = train_test_split(
		X_train, y_train,
		train_size=percentage,
		stratify=y_train,
		random_state=RANDOM_SEED
	)
      
	print(f"Reduced Train shape: {X_train.shape} ({len(X_train)/len(X_raw):.1%})")
      
else:
    
    print("Training set size is less than or equal to the new size. No reduction applied.")
    
	# end if

---

## Preprocessing

In questa sezione vengono raccolte le principali strategie di preprocessing applicate ai biflussi del dataset prima della fase di training. L'obiettivo è trasformare i dati grezzi in una rappresentazione più adatta ai modelli classici e ibridi quantistici. Queste strategie consentono di confrontare diverse modalità di preparazione dei dati, da approcci più semplici e generici a soluzioni più mirate al dominio del traffico di rete.

Le feature originali considerate sono:

- `DIR`: direzione del pacchetto
- `PL`: packet length
- `TCPWIN`: finestra TCP
- `IAT`: inter-arrival time

**Strategies**

**1. Masking del padding + Log1p normalization:**
Approccio guidato dal dominio applicativo. I pacchetti di padding vengono identificati e azzerati per evitare che influenzino il modello. Successivamente viene applicata una trasformazione `Log1p` alle feature numeriche (`PL`, `TCPWIN`, `IAT`) per comprimere il range dinamico e ridurre l'effetto degli outlier.

**2. Min-Max Scaling standard:**
Approccio generico che applica una normalizzazione lineare nell'intervallo `[0, 1]` a tutte le feature. Non gestendo esplicitamente il padding, questa strategia può trattare i valori fittizi come dati reali e risultare sensibile agli outlier.

**3. Fusione DIR/PL + Min-Max Scaling:**
La direzione (`DIR`) e la lunghezza (`PL`) vengono combinate in una singola feature con segno, così da rappresentare il traffico in modo più compatto. Dopo questa trasformazione, il numero di feature passa da 4 a 3 e viene applicato un Min-Max Scaling standard.

**4. Masking del padding + Log1p + fusione DIR/PL:**
Strategia ibrida che unisce i vantaggi della Strategy 1 e della Strategy 3. Prima gestisce correttamente il padding e applica `Log1p`, poi combina `DIR` e `PL` in una feature con segno. In questo modo si ottiene una rappresentazione più compatta, semanticamente coerente e generalmente più robusta.

In [ ]:
PREPROCESSING_REGISTRY = {
	"Log1p": log1pPreprocessing,
	"MinMax": minMaxPreprocessing,
	"MinMax-DirPL": lambda X: minMaxPreprocessing(X, combine_dir_pl_flag=True),
	"Log1p-DirPL": lambda X: log1pPreprocessing(X, combine_dir_pl_flag=True),
}
""" Specifies the preprocessing strategy to apply to the dataset. """

PREPROCESSING_STRATEGY = "MinMax"

if PREPROCESSING_STRATEGY not in PREPROCESSING_REGISTRY:
	raise ValueError(
		f"PREPROCESSING_STRATEGY deve essere uno tra: " + 
		f"{list(PREPROCESSING_REGISTRY.keys())}"
	)

preprocess_fn = PREPROCESSING_REGISTRY[PREPROCESSING_STRATEGY]

print(f"Applying preprocessing strategy {PREPROCESSING_STRATEGY}...")

X_train_proc = preprocess_fn(X_train)
X_val_proc = preprocess_fn(X_val)
X_test_proc = preprocess_fn(X_test)

# Aggiorna il numero di feature in base alla strategia scelta.
N_FEATURES = X_train_proc.shape[2]

print("\nPreprocessing complete.")

print(f"\nTrain shape: {X_train_proc.shape}")
print(f"Val shape:   {X_val_proc.shape}")
print(f"Test shape:  {X_test_proc.shape}")

print(f"\nAdjusted N_FEATURES: {N_FEATURES}")

---

## Dataset & Dataloader

Un `DataLoader` è un oggetto che preleva campioni dal dataset e genera batch in modo efficiente.

Gli iperparametri principali del `DataLoader` sono:
-	**Batch size** (cioè il numero di campioni in un mini-batch).

	Utilizzando la GPU, un batch size più grande rende l'addestramento più efficiente. Tuttavia, un batch size più piccolo può portare a risultati migliori in termini di accuratezza finale.
	***La selezione del batch size appropriato e di altri iperparametri è fondamentale per l'ottimizzazione del modello e dipende dalle caratteristiche specifiche del dataset e dell'architettura del modello.***

-	**Number of workers**

	Questo iperparametro determina il numero di processi paralleli utilizzati per caricare i dati. Un numero maggiore di worker può accelerare il caricamento dei dati, ma può anche aumentare l'uso della memoria e la complessità del sistema.
	***È buona norma impostarli in base al numero di core della CPU.***

-	**Shuffle**

	Se impostato su `True`, i dati vengono mescolati ad ogni epoca, garantendo che il modello non impari sequenze specifiche dei dati. Si può impostare questo iperparametro per evitare overfitting e migliorare la generalizzazione del modello (solo per la fase di training).

-	**Drop last**

	Se impostato su `True`, l'ultimo batch viene scartato se non contiene il numero completo di campioni. Questo può essere utile per garantire che tutti i batch abbiano la stessa dimensione, semplificando l'addestramento del modello.
	***Si consiglia di impostarlo su `True` per il training e su `False` per la validazione e il test, perché in questi ultimi casi è importante valutare il modello su tutti i campioni disponibili.***

In [ ]:
import importlib
importlib.reload(constants)
from constants import BATCH_SIZE

#   ####################################################################    #

# Si utilizza la classe dedicata MirageDataset per creare i dataset di PyTorch,
# in modo che siano compatibili con le reti neurali convoluzionali 1D (CNN1D).
train_dataset = MirageDataset(X_train_proc, y_train)
val_dataset = MirageDataset(X_val_proc, y_val)
test_dataset = MirageDataset(X_test_proc, y_test)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
	num_workers=0,
    shuffle=True, drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
	num_workers=0,
    shuffle=False, drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
	num_workers=0,
    shuffle=False, drop_last=False
)

---

## Model Selection

### da models/nn_models.py

-	**Amplitude Embedding Model** (`AmplitudeEmbedding`)

	Una rete neurale ibrida che sfrutta l'Amplitude Embedding per codificare i dati classici nelle ampiezze dello stato quantistico. Questo approccio massimizza la densità dei dati, consentendo la codifica di $2^N$ features in $N$ qubit, preceduta da uno strato denso classico attivato da una funzione sigmoide.

-	**Angle Embedding Model** (`AngleEmbedding`)

	Un modello ibrido semplificato che utilizza l'Angle Embedding, dove le feature di input vengono mappate direttamente sugli angoli di rotazione dei qubit in un rapporto 1:1. Presenta uno strato di pre-elaborazione classico seguito da un circuito quantistico con strati fortemente entangled, offrendo una strategia di embedding shallow e resistente al rumore.

-	**Ring Model** (`RingEmbedding`)

	Un'architettura ibrida che impiega una strategia di Ring Embedding personalizzata, suddividendo l'input in due set di feature codificate tramite rotazioni e schemi di entanglement circolari CNOT. Questo design raddoppia la capacità dei dati rispetto al semplice angle embedding e introduce correlazioni tra i qubit già dalle prime fasi del circuito.

-	**Waterfall Model** (`WaterfallEmbedding`)

	Un modello ibrido complesso che presenta uno schema di Waterfall Embedding, suddividendo gli input in blocchi di rotazione Y e Z. Integra una connettività densa e all-to-all di porte CNOT nella prima fase, creando uno stato altamente entangled prima degli strati del variational ansatz.

-	**Classical 1D CNN Model** (`TrafficCNN`)

	Un modello classico basato su una rete neurale convoluzionale 1D, progettata per estrarre caratteristiche locali dai dati sequenziali. Questo approccio sfrutta strati convoluzionali e di pooling per catturare pattern temporali o spaziali nei dati di input.

### da models/complex_hybrid_models.py

-	**AmpCnn Model** (`AmpCnn`)

	Un modello ibrido che combina l'Amplitude Embedding con una rete neurale convoluzionale 1D, sfruttando le capacità di codifica quantistica per migliorare l'estrazione delle caratteristiche locali dai dati sequenziali.

-	**Classical Twin Model** (`ClassicalTwin`)

	Un modello classico basato su una rete neurale a due rami, progettata per elaborare due flussi di dati paralleli e combinare le informazioni estratte per migliorare le prestazioni predittive.

-	**Classical Light Model** (`ClassicalLight`)

	Un modello classico leggero, progettato per essere efficiente in termini di risorse computazionali e memoria, pur mantenendo buone prestazioni predittive.

-	**CnnAmpCnn Model** (`CnnAmpCnn`)

	Un modello ibrido **CNN–Quantum–CNN**: una prima CNN 1D estrae feature locali dai pacchetti, che vengono proiettate (con layer denso + sigmoide) nello spazio richiesto dall’**Amplitude Embedding**. L’output del circuito quantistico (con strati fortemente entangled) viene poi raffinato da una seconda CNN 1D e da layer fully connected per la classificazione finale multiclasse.

-	**Dense Model** (`Dense`)

	Un modello classico basato su una rete neurale fully connected, progettato per elaborare input di dimensioni fisse e catturare relazioni complesse tra le feature, grazie a strati densi e funzioni di attivazione non lineari.

In [ ]:
import importlib
import sys
importlib.reload(constants)
from constants import (
    MODEL_REGISTRY,
    N_QUBITS, N_LAYERS,
    N_FEATURES, N_PACKETS
)

#   ####################################################################    #

SELECTED_MODEL = "AmplitudeEmbedding"

if SELECTED_MODEL not in MODEL_REGISTRY:
    available = ", ".join(MODEL_REGISTRY.keys())
    raise ValueError(f"Modello '{SELECTED_MODEL}' non valido.\n\nOpzioni: {available}")

	# end if

# Import dinamico del modulo e della classe del modello selezionato.
module_name, class_name = MODEL_REGISTRY[SELECTED_MODEL]
sys.path.append(os.path.abspath('..'))
clean_module_name = module_name.lstrip('../').replace('/', '.')
module = importlib.import_module(clean_module_name)

# Si istanzia il modello selezionato con i parametri specificati.
HybridModel = getattr(module, class_name)
model = HybridModel(
    n_qubits=N_QUBITS,
    n_layers=N_LAYERS,
    n_features=N_FEATURES,
    n_packets=N_PACKETS,
    num_classes=NUM_CLASSES
)

# Si sposta il modello sul dispositivo corretto (CPU, GPU o MPS)
# e si imposta il tipo di dato appropriato.
if DEVICE == 'mps':
	model = model.to(DEVICE).float()
else:
	model = model.to(DEVICE).double()

print(f"Modello selezionato: {SELECTED_MODEL} -> {class_name}")
print(model.get_model_name())
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Totale parametri addestrabili: {total_params:,}")

---

## Training Setup

Durante il training, due componenti sono fondamentali:

- **Optimizer**: è l’algoritmo che aggiorna i pesi del modello dopo ogni batch, usando i gradienti calcolati con la backpropagation (e.g. in pratica decide come e quanto modificare i parametri per ridurre l’errore). Pertanto, la scelta dell’optimizer (e del **learning rate**) influenza stabilità, velocità di convergenza e qualità finale delle predizioni.

	Le scelte più comuni sono:

	- `SGD`

		Ottimizzatore “classico”: aggiorna i pesi con passi uniformi (spesso con `momentum`).  
		È adatto quando si vuole un comportamento semplice e controllabile, e funziona bene su modelli/dataset stabili (con tuning accurato del learning rate).

	- `Adam`

		In generale, `Adam` è preferito come scelta di default perché usa learning rate adattivi e converge più rapidamente su problemi complessi o con poco tuning.

	- `RMSprop`, `AdamW`

		`RMSprop` è spesso utile con gradienti rumorosi/non stazionari (es. dati sequenziali), mentre `AdamW` è indicato quando si vuole una regolarizzazione migliore grazie al weight decay disaccoppiato.

- **Loss function (`criterion`)**: è la funzione che misura quanto le predizioni del modello sono lontane dai target reali. Fornisce il segnale da minimizzare durante l’addestramento.

	Nel problema di classificazione multiclasse spesso si usano:

	- `CrossEntropyLoss`

		È la scelta standard per la **classificazione multiclasse**, che misura la differenza tra la distribuzione di probabilità predetta e la distribuzione reale delle etichette. Penalizza fortemente le predizioni molto sicure ma sbagliate, guidando il modello a migliorare le sue predizioni minimizzando questa differenza durante l'addestramento. È adatta quando il dataset è **abbastanza bilanciato** o quando non si vuole introdurre un trattamento diverso tra classi.

	- `Weighted CrossEntropy`

		Stessa idea della CrossEntropy standard, ma con `weight=class_weights`. Assegna pesi diversi a ciascuna classe in base alla loro frequenza nel training set. Le classi con meno campioni ricevono pesi più alti, aiutando il modello a prestare maggiore attenzione durante l'addestramento. È indicata per dataset con uno **sbilanciamento moderato**.

	- `Focal Loss`

		La Focal Loss è progettata per affrontare dataset **fortemente sbilanciati**, riducendo il peso degli esempi facili e concentrandosi maggiormente su quelli difficili o mal classificati. È utile quando il modello tende a favorire troppo le classi più frequenti. I pesi vengono calcolati nello stesso modo della Weighted CrossEntropy.
		
		È necessario definire due parametri: `alpha` serve a bilanciare le classi, mentre `gamma` è un parametro di focalizzazione regolabile che determina quanto velocemente gli esempi facili vengono ridotti di peso.


In [ ]:
import importlib
importlib.reload(constants)
from constants import (
	LEARNING_RATE,
	LOSS_REGISTRY
)

#   ####################################################################    #

SELECTED_LOSS = "CrossEntropy"

# Per completare la configurazione della fase di training,
# si definisce l'optimizer e la funzione di loss.
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

loss_dtype = torch.float if DEVICE == 'mps' else torch.double
loss_params = LOSS_REGISTRY[SELECTED_LOSS]["params"]

if SELECTED_LOSS == "CrossEntropy":
	criterion = nn.CrossEntropyLoss()

elif SELECTED_LOSS == "WeightedCrossEntropy":
	class_weights = compute_class_weights(
		y_train,
		NUM_CLASSES
	).to(DEVICE).to(loss_dtype)

	criterion = nn.CrossEntropyLoss(weight=class_weights)

elif SELECTED_LOSS == "Focal":
	alpha = loss_params.get("ALPHA", "class_weights")
	gamma = loss_params.get("GAMMA", 2.0)

	if alpha == "class_weights":
		alpha = compute_class_weights(y_train, NUM_CLASSES)
	elif alpha == "uniform":
		alpha = torch.ones(NUM_CLASSES)
	elif alpha == "custom":
		alpha = torch.tensor(alpha)
	else:
		raise ValueError(f"Invalid alpha_mode: {alpha}. Must be 'class_weights', 'uniform', or 'custom'.")

	alpha = alpha.to(DEVICE).to(loss_dtype)

	criterion = torch.hub.load(
		'adeelh/pytorch-multi-class-focal-loss',
		model='focal_loss',
		alpha=alpha,
		gamma=gamma,
		reduction='mean',
		device=DEVICE,
		dtype=loss_dtype,
		force_reload=False
	)

else:
	raise ValueError(
		f"Tipo di loss non supportato: {SELECTED_LOSS}. "
		"Usare: 'CrossEntropy', 'WeightedCrossEntropy' oppure 'Focal'."
	)

print(f"Loss selezionata: {SELECTED_LOSS}")

## Training Loop

In [ ]:
import importlib
importlib.reload(constants)
from constants import (
	EPOCHS, PATIENCE, EARLY_STOPPING
)

#   ####################################################################    #

# Questo dizionario terrà traccia delle metriche
# di addestramento e validazione per ogni epoca.
history = {
    'accuracy': [],
    'val_accuracy': [],
    'loss': [],
    'val_loss': []
}

# Inizializza le informazioni per il salvataggio del modello e dei risultati.
id = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
model_name = f"{len(history['loss'])}E_{SELECTED_LOSS}_{PREPROCESSING_STRATEGY}_{SELECTED_MODEL}"
output_dir = f"../models/{id}/"

os.makedirs(f"{output_dir}", exist_ok=True)

print(
    f"[INFO] Addestramento iniziato.\n" +
	f"[DATETIME] {id}\n" +
	f"[NAME] {model_name}"
)

# Inizializza le variabili per il monitoraggio della miglior loss di validazione.
best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
patience_counter = 0

# Inizia il training loop per il numero di epoche specificato.
start_time = time.time()
for epoch in range(EPOCHS):
    
    start_epoch_time = time.time()
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    
	# Calcola la durata dell'epoca corrente.
    end_epoch_time = time.time()
    epoch_duration = end_epoch_time - start_epoch_time

	# Aggiorna lo storico delle metriche per l'epoca corrente.
    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

	# Stampa in console le metriche dell'epoca corrente.
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Loss: {train_loss:.4f} - Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}"
          f" | Time: {epoch_duration:.2f}s")

	# CHECK : se la loss di validazione migliora,
	# salva il modello e resetta il contatore per la patience.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"  -> Validation loss improved. Model saved.")
    else:
        patience_counter += 1
        print(f"  -> No improvement. Patience: {patience_counter}", f"/ {PATIENCE}" if EARLY_STOPPING else "")

	# Se il contatore di patience raggiunge il limite
	# e l'early stopping è abilitato, si interrompe il training.
    if patience_counter >= PATIENCE and EARLY_STOPPING:
        print("Early stopping triggered.")
        break

# Calcola e stampa il tempo totale di addestramento.
total_time = time.time() - start_time
print(f"\nTraining complete in {total_time/60:.2f} minutes.")

# Salva i pesi del modello con la miglior loss di validazione.
last_model = copy.deepcopy(model)
model.load_state_dict(best_model_wts)

In [ ]:
id = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
model_name = f"{len(history['loss'])}E_{SELECTED_LOSS}_{PREPROCESSING_STRATEGY}_{SELECTED_MODEL}"
output_dir = f"../models/{id}/"

os.makedirs(f"{output_dir}/{model_name}", exist_ok=True)

# with open(f"{output_dir}/{model_name}/model_summary.txt", "w") as f:
#     model.summary(print_fn=lambda x: f.write(x + "\n"))

# Salva lo storico delle metriche di addestramento e validazione in un file CSV.
df_history = pd.DataFrame(history)
df_history.to_csv(f"{output_dir}/{model_name}/training_history.csv", index=False)

# Salva i pesi del modello addestrato in un file .PTH
torch.save(model.state_dict(), f"{output_dir}/{model_name}/model.pth")

---

## Testing and Evaluation

In [ ]:
# Questo blocco di codice serve per caricare un modello
# precedentemente addestrato e salvato, insieme al suo storico di addestramento.
# Viene specificata la directory di output e il nome del modello da caricare.
# Il modello viene quindi istanziato con i parametri appropriati e
# i pesi salvati vengono caricati nel modello.

output_dir = "../models/2026-07-01_12-29"
model_name = "50E_Focal_Log1p-DirPL_CnnAmpCnn"

model = HybridModel(
    n_qubits=N_QUBITS,
    n_layers=N_LAYERS,
    n_features=N_FEATURES,
    n_packets=N_PACKETS,
    num_classes=NUM_CLASSES
)

if DEVICE == 'mps':
    model = model.to(DEVICE).float()
else:
    model = model.to(DEVICE).double()
    
weights = torch.load(f"{output_dir}/{model_name}/model.pth", map_location=DEVICE)
load_result = model.load_state_dict(weights)

print(f"Esito caricamento: {load_result}")
model = model.to(DEVICE)
model.eval()

print('Model loaded correctly')

# Si recupera lo storico delle metriche di addestramento e validazione dal file CSV salvato.
df_history = pd.read_csv(f"{output_dir}/{model_name}/training_history.csv")
history = df_history.to_dict(orient='list')

### Valutazione dell'accuracy sul test set

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

### Grafici della funzione di loss e di accuracy in fase di training e validazione

In [ ]:
acc = history['accuracy']
val_acc = history['val_accuracy']
loss = history['loss']
val_loss = history['val_loss']
epochs_range = range(len(acc))

training_validation_plots = plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, acc, label='Train Acc')
plt.plot(epochs_range, val_acc, label='Val Acc')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

training_validation_plots.savefig(f"{output_dir}/{model_name}/train_val_plots.png")

### Matrice di confusione

Una matrice di confusione è una tabella che riassume le prestazioni di un modello di classificazione, mettendo a confronto le classi reali (righe) con le classi predette dal modello (colonne). Ogni cella `(i,j)` indica quante osservazioni della classe reale `i` sono state classificate come classe predetta `j`, permettendo di vedere non solo quante previsioni sono corrette, ma anche quali errori specifici il modello commette.

In [ ]:
model.eval()
all_preds = []
all_labels = []

# Si utilizza "torch.no_grad()" per disabilitare
# il calcolo del gradiente durante la fase di valutazione,
# migliorando le prestazioni e riducendo l'uso della memoria.
with torch.no_grad():
    for inputs, labels in test_loader:
        
		# Sposta i dati sul DEVICE selezionato prima della predizione.
        inputs = inputs.to(DEVICE)
        
		# Calcola le predizioni del modello per il batch corrente.
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        
		# Infine, si utilizza "extend" per aggiungere le predizioni e le etichette
        # del batch corrente alle liste globali "all_preds" e "all_labels".
        # A differenza di "append", che aggiunge un singolo elemento,
		# "extend" aggiunge tutti gli elementi di un Iterable alla lista esistente.
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
		# ATTENZIONE!
		# Sostituire con "all_labels.extend(labels.numpy())" se si utilizza la CPU.
    
		# end for inputs, labels

cm = confusion_matrix(all_labels, all_preds, normalize='true')

confusion_matrix_plot = plt.figure(figsize=(10, 8))

ax = sns.heatmap(
	cm,
	annot=False,
	fmt='',
	cmap='plasma_r',   # Mappa colori plasma invertita
	linewidths=0.5,    # Spessore linee della griglia
	mask= cm == 0,     # Applica la maschera
	linecolor='black', # Colore linee della griglia
	square=True,       # Celle quadrate
	cbar_kws={ "ticks": [0.1, 1, 10, 100] },
	xticklabels=le.classes_,
	yticklabels=le.classes_
)

ax.set_facecolor('white')

empty_cols = np.where(cm.sum(axis=0) == 0)[0]

# Se ci sono colonne vuote, aggiunge un simbolo "●" al centro di ciascuna cella vuota.
for col in empty_cols:
    for row in range(cm.shape[0]):
        
        # Per centrare il testo nella cella, si aggiunge 0.5 a "row" e "col".
        ax.text(
            col + 0.5, row + 0.5, '●',
            ha='center', va='center',
            color='black', fontsize=15
		)
    
		# end for row
	# end for col

plt.title("Confusion Matrix")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

confusion_matrix_plot.savefig(f"{output_dir}/{model_name}/confusion_matrix.png")

### Report di classificazione completo

In [ ]:
report_dict = classification_report(
    all_labels, all_preds,
    target_names=le.classes_, labels=np.arange(NUM_CLASSES),
    digits=4, zero_division=0
)

with open(f"{output_dir}/{model_name}/classification_report.txt", "w") as f:
    f.write(report_dict)